# weewx / console / battery

**Intent:** For an input console battery voltage (`weatherstation_console_battery_voltage`, volts), output the remaining capacity (`weewx_console_battery_capacity_pct`, percent from 0 to 100 in steps of 5), so the console battery can be tracked without a gauge.

**Notes:**
- Purpose: Turn a battery voltage reading into a remaining capacity so the console battery can be tracked without a gauge
- Flavour: `pyfunc`, a Python model that keeps the exact arctan; `wasm` if an embedded consumer appears
- Data splits: `training` the 2024 run, digitised from a plot, which with the incumbent's 36 control points is what the arctan is fitted to; `validation` the 2026 run from InfluxDB; `backtest` the 2026 readings labelled by the incumbent Home Assistant curve (agreement with what runs today, not battery truth); `golden` the fixed voltages and expected percentages that become `golden.json`

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, version and dataset names as constants (`model.md` 6.6).

In [ ]:
from workbench.data import data_path, golden_data, query_influxdb3, read_data
from workbench.env import init_env
from workbench.plot import show_plot
from workbench.project import load_signature, provenance, record_project, scratch_project_dir, signature_names

env = init_env()

WORKSPACE = "weewx/console/battery"
MODEL_NAME = "weewx_console_battery"
TARGETS = [{"kind": "served"}]
signature = load_signature(WORKSPACE)
INPUTS, OUTPUTS = signature_names(signature)
TRAINING = "console_battery_voltage_2024-06-06_2024-10-25"
VALIDATION = "console_battery_voltage_2026-08-13_2026-10-30"
GOLDEN = "console_battery_voltage_golden"

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** query InfluxDB for the `validation` file only when it is absent (`model.md` 6.5), then load `training` and `validation` through `data_path`.

In [ ]:
query_influxdb3(
    env,
    """
SELECT time, value
FROM sensor__voltage
WHERE module = 'homeassistant'
  AND entity_id = 'weatherstation_console_battery_voltage'
  AND time >= '2026-08-13T00:00:00Z'
  AND time < '2026-10-31T00:00:00Z'
ORDER BY time
""",
    WORKSPACE,
    "validation",
    VALIDATION,
)

import polars as pl

data_training_path = data_path(WORKSPACE, "training", TRAINING)
data_raw_pl = read_data(WORKSPACE, "training", TRAINING).rename({"time": "Timestamp", "voltage": "Voltage"}).with_columns(pl.col("Voltage").cast(pl.Float32))
data_validation_pl = read_data(WORKSPACE, "validation", VALIDATION).rename({"time": "Timestamp", "value": "Voltage"}).with_columns(pl.col("Voltage").cast(pl.Float32))

## Explore

- **Stage:** 2, explore.
- **Intent:** plot the raw, sampled and normalised voltage to see the discharge shape. Plain Python is allowed here and nothing reported from it is shipped (R4).

In [ ]:
show_plot("Raw", data_raw_pl, "Timestamp", "Voltage")

In [ ]:
data_sampled_pl = data_raw_pl.sort("Timestamp").group_by_dynamic("Timestamp", every="10m").agg(pl.col("Voltage").min())
show_plot("Raw", data_raw_pl, "Timestamp", "Voltage")
show_plot("Sampled", data_sampled_pl, "Timestamp", "Voltage", "o")

In [ ]:
data_normalised_pl = data_sampled_pl.drop("Timestamp").with_row_index("Capacity")
data_normalised_pl = data_normalised_pl.with_columns((100 - pl.col("Capacity") / pl.max("Capacity") * 100).alias("Capacity"))
show_plot("Normalised", data_normalised_pl, "Voltage", "Capacity", "o")

## Fit

- **Stage:** 3, fit.
- **Intent:** fit a bounded arctan from voltage to capacity, with a polynomial for comparison. The arctan is fitted to the `training` run and to the 36 control points of the incumbent Home Assistant curve, with the two sources weighted equally in total, so the model starts from what runs today and from the full discharge. The `validation` and `backtest` runs are not used to fit (R6).

In [ ]:
from math import atan

import numpy as np
from scipy.optimize import curve_fit

y_min = 0.0
y_max = 100.0
y_data = data_normalised_pl.select("Capacity").to_numpy().flatten()
x_min = data_normalised_pl.select(pl.min("Voltage")).item()
x_max = data_normalised_pl.select(pl.max("Voltage")).item()
x_data = data_normalised_pl.select("Voltage").to_numpy().flatten()
incumbent_points = np.array(
    [
        [4, 100],
        [3.97, 99],
        [3.95, 98],
        [3.92, 96],
        [3.91, 94],
        [3.9, 93],
        [3.89, 91],
        [3.88, 88],
        [3.87, 87],
        [3.85, 83],
        [3.83, 81],
        [3.81, 65],
        [3.79, 59],
        [3.78, 56],
        [3.76, 50],
        [3.75, 49],
        [3.74, 43],
        [3.72, 38],
        [3.7, 35],
        [3.68, 33],
        [3.66, 28],
        [3.64, 22],
        [3.62, 21],
        [3.61, 18],
        [3.59, 16],
        [3.58, 15],
        [3.56, 10],
        [3.54, 9],
        [3.52, 8],
        [3.49, 7],
        [3.48, 6],
        [3.46, 5],
        [3.44, 3],
        [3.4, 2],
        [3.38, 1],
        [3.32, 0],
    ]
)
x_fit = np.concatenate([x_data, incumbent_points[:, 0]])
y_fit = np.concatenate([y_data, incumbent_points[:, 1]])
sigma_fit = np.concatenate([np.ones(len(x_data)), np.full(len(incumbent_points), (len(incumbent_points) / len(x_data)) ** 0.5)])

fit_poly_coefficients = np.polyfit(x_data, y_data, 2)
fit_poly_function = np.poly1d(fit_poly_coefficients)


def fit_hass_poly_function(x):
    if x < x_min:
        return y_min
    if x > x_max:
        return y_max
    return fit_poly_function(x)


def fit_custom_bounded_poly_function(x):
    if x < x_min:
        return y_min
    if x > x_max:
        return y_max
    y = fit_poly_function(x)
    if y < y_min:
        return y_min
    if y > y_max:
        return y_max
    return y


def fit_custom_arctan(x, k, w, x0, y0):
    return k * np.arctan(w * (x - x0)) + y0


fit_arctan_coefficients = curve_fit(fit_custom_arctan, x_fit, y_fit, p0=[1, 1, 0.5 * (x_max + x_min), 0.5 * (y_max + y_min)], sigma=sigma_fit, full_output=True)[0]


def fit_custom_bounded_arctan(x):
    if x < x_min:
        return y_min
    if x > x_max:
        return y_max
    y = fit_arctan_coefficients[0] * atan(fit_arctan_coefficients[1] * (x - fit_arctan_coefficients[2])) + fit_arctan_coefficients[3]
    if y < y_min:
        return y_min
    if y > y_max:
        return y_max
    return float(int(y) // 5 * 5)


scale_factor = 1000
data_fitted_pl = pl.arange(2.9 * scale_factor, 4.1 * scale_factor, eager=True).to_frame("Voltage").with_columns(pl.col("Voltage") / scale_factor)
data_fitted_hass_poly_pl = data_fitted_pl.with_columns(pl.col("Voltage").map_elements(lambda x: fit_hass_poly_function(x), return_dtype=pl.Float64).alias("Capacity"))
data_fitted_custom_bounded_poly_pl = data_fitted_pl.with_columns(pl.col("Voltage").map_elements(lambda x: fit_custom_bounded_poly_function(x), return_dtype=pl.Float64).alias("Capacity"))
data_fitted_custom_bounded_step_arctan_pl = data_fitted_pl.with_columns(pl.col("Voltage").map_elements(lambda x: fit_custom_bounded_arctan(x), return_dtype=pl.Float64).alias("Capacity"))

show_plot("Normalised", data_normalised_pl, "Voltage", "Capacity", "o")
show_plot("HASS Poly", data_fitted_hass_poly_pl, "Voltage", "Capacity", ":")
show_plot("Custom Bounded Poly", data_fitted_custom_bounded_poly_pl, "Voltage", "Capacity", ":")
show_plot("Custom Bounded Step Arctan", data_fitted_custom_bounded_step_arctan_pl, "Voltage", "Capacity", "-")

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** write `model.py` with the fitted attributes built in, check the shipped function equals the fitted function and check the predictions on the `validation` run stay in range (R4), then run the acceptance check the notebook defines (R14). The `backtest` run is the 2026 readings labelled by the incumbent, the degree 3 Home Assistant `compensation` curve through its 36 data points, clamped to 0 to 100 and rounded, so the check measures agreement with what runs today and not the battery's true capacity. Two plots show the accuracy: the shipped curve against the incumbent labels and the training run, and the error at each backtest voltage. The verdict is recorded and blocks release, not the run.

In [ ]:
import importlib.util
from os.path import join

from workbench.config import NOTEBOOK_DIR

MODEL_TEMPLATE = """from math import atan

import pandas as pd
from mlflow.models import set_model
from mlflow.pyfunc import PythonModel

INPUT = "weatherstation_console_battery_voltage"
OUTPUT = "weewx_console_battery_capacity_pct"
X_MIN = {x_min!r}
X_MAX = {x_max!r}
Y_MIN = {y_min!r}
Y_MAX = {y_max!r}
STEP = 5
K = {k!r}
W = {w!r}
X0 = {x0!r}
Y0 = {y0!r}


def capacity_pct(voltage):
    if voltage < X_MIN:
        return Y_MIN
    if voltage > X_MAX:
        return Y_MAX
    capacity = K * atan(W * (voltage - X0)) + Y0
    if capacity < Y_MIN:
        return Y_MIN
    if capacity > Y_MAX:
        return Y_MAX
    return float(int(capacity) // STEP * STEP)


class ConsoleBattery(PythonModel):
    def predict(self, context, model_input: pd.DataFrame, params=None):
        return pd.DataFrame({{OUTPUT: [capacity_pct(voltage) for voltage in model_input[INPUT]]}})


set_model(ConsoleBattery())
"""

model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.py")
with open(model_path, "w") as model_file:
    model_file.write(
        MODEL_TEMPLATE.format(
            x_min=float(x_min),
            x_max=float(x_max),
            y_min=y_min,
            y_max=y_max,
            k=float(fit_arctan_coefficients[0]),
            w=float(fit_arctan_coefficients[1]),
            x0=float(fit_arctan_coefficients[2]),
            y0=float(fit_arctan_coefficients[3]),
        )
    )
model_spec = importlib.util.spec_from_file_location("model", model_path)
model = importlib.util.module_from_spec(model_spec)
model_spec.loader.exec_module(model)
capacity_pct = model.capacity_pct

assert all(capacity_pct(v) == fit_custom_bounded_arctan(v) for v in np.linspace(x_min - 0.2, x_max + 0.2, 500)), "the shipped function differs from the fitted function"

validation_capacity = [capacity_pct(v) for v in data_validation_pl["Voltage"].to_list()]
validation_rises = sum(1 for a, b in zip(validation_capacity, validation_capacity[1:], strict=False) if b > a)
print("validation samples", len(validation_capacity), "capacity range", min(validation_capacity), max(validation_capacity), "step rises on a one-way discharge", validation_rises)
assert min(validation_capacity) >= 0.0 and max(validation_capacity) <= 100.0

In [ ]:
from glob import glob

import pandas as pd

from workbench.model import evaluate_acceptance, measure

BACKTEST = sorted(glob(join(NOTEBOOK_DIR, WORKSPACE, "data", "backtest", "*", "*.csv")))


def acceptance_checks():
    backtest = pd.read_csv(BACKTEST[-1])
    shipped = [capacity_pct(voltage) for voltage in backtest["weatherstation_console_battery_voltage"]]
    measured = measure(backtest["incumbent_capacity_pct"], shipped)
    return [{"split": "backtest", "output": "weewx_console_battery_capacity_pct", "name": "rmse", "value": measured["rmse"], "limit": 5, "direction": "max"}]


evaluation = evaluate_acceptance(acceptance_checks, {"backtest": BACKTEST})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "passed", entry["passed"])
print("acceptance", "passed" if evaluation["passed"] else "failed, the shipped model differs from the incumbent curve by more than the limit")

In [ ]:
data_shipped_pl = data_fitted_pl.with_columns(pl.col("Voltage").map_elements(capacity_pct, return_dtype=pl.Float64).alias("Capacity"))
data_backtest_pl = pl.read_csv(BACKTEST[-1]).rename({"weatherstation_console_battery_voltage": "Voltage", "incumbent_capacity_pct": "Incumbent"})
data_backtest_pl = data_backtest_pl.with_columns(pl.col("Voltage").map_elements(capacity_pct, return_dtype=pl.Float64).alias("Shipped")).with_columns(
    (pl.col("Shipped") - pl.col("Incumbent")).alias("Error")
)

show_plot("Training", data_normalised_pl, "Voltage", "Capacity", ".")
show_plot("Incumbent", data_backtest_pl, "Voltage", "Incumbent", "o")
show_plot("Shipped", data_shipped_pl, "Voltage", "Capacity", "-")

In [ ]:
show_plot("Shipped minus incumbent", data_backtest_pl, "Voltage", "Error", "o")

## Package

- **Stage:** 5, package.
- **Intent:** write the `golden` file from the shipped `model.py`, build `signature.json` and `golden.json` from it, copy `model.py` and the evaluation into the project and verify it (R1, R3, R4, R14).

In [ ]:
import json

from workbench.model import package_project, register_preview
from workbench.replay import require_replay


def golden_samples():
    voltages = [round(v, 4) for v in [x_min - 0.1, x_min, *np.linspace(x_min, x_max, 40).tolist(), x_max, x_max + 0.1]]
    return {
        "weatherstation_console_battery_voltage": [*voltages, None],
        "weewx_console_battery_capacity_pct": [*[capacity_pct(v) for v in voltages], None],
        "expect": ["ok"] * len(voltages) + ["error"],
    }


golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "pyfunc",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "mlflow", "weewx_console_battery_voltage, training split", [data_training_path, golden_path], "arctan fit, steps of 5", implementation="mlflow.pyfunc model.py, requires mlflow and pandas"
    ),
    model_path=model_path,
    evaluation=evaluation,
    sequence="across the fitted voltage range and beyond it",
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped runtime")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version, "with alias preview, set release by hand after review")